# 01 - Exploratory Data Analysis
Official **20 Newsgroups** dataset (via scikit-learn). Run `python -m src.data_loader` once before this notebook.
All conclusions below must be written from the outputs you obtain - none are pre-filled.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import load_config, resolve
cfg = load_config()
FIG = resolve(cfg, "figures_dir"); FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
from src.data_loader import load_processed_dataset
df = load_processed_dataset(cfg)
print("Dimensions:", df.shape)
print("Documents:", len(df), "| Categories:", df.category.nunique())
df[["document_id","dataset_split","category","category_id"]].head()

## Documents per category and class distribution

In [ ]:
counts = df.category.value_counts().sort_index()
display(counts.to_frame("documents"))
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=counts.values, y=counts.index, color="#3949ab", ax=ax)
ax.set_title("Documents per Category (all splits)"); ax.set_xlabel("Number of documents"); ax.set_ylabel("Category")
plt.tight_layout(); fig.savefig(FIG / "eda_category_distribution.png", dpi=150); plt.show()
split_ct = df.groupby(["category","dataset_split"]).size().unstack(fill_value=0)
display(split_ct)

## Class imbalance analysis

In [ ]:
print("Largest/smallest class ratio: %.2f" % (counts.max()/counts.min()))
print("Largest:", counts.idxmax(), counts.max(), "| Smallest:", counts.idxmin(), counts.min())
print("Coefficient of variation: %.3f" % (counts.std()/counts.mean()))

## Missing, empty and duplicate documents

In [ ]:
print("Missing values:\n", df[["document_text","category","category_id"]].isna().sum())
print("Empty raw:", (df.document_text.str.strip()=="").sum())
print("Empty after header/footer/quote removal:", (df.text_no_metadata.str.strip()=="").sum())
print("Empty after cleaning:", df.is_empty_after_cleaning.sum())
print("Exact duplicate raw documents:", df.document_text.duplicated().sum())
print("Duplicate cleaned documents:", df.is_duplicate_cleaned_text.sum())
dups = df[df.is_duplicate_cleaned_text]
display(dups.groupby("category").size().sort_values(ascending=False).head(10).to_frame("duplicates"))

## Document length and word count

In [ ]:
df["n_chars_raw"] = df.document_text.str.len()
df["n_chars_clean"] = df.cleaned_text.str.len()
df["n_words_clean"] = df.cleaned_text.str.split().str.len()
display(df[["n_chars_raw","n_chars_clean","n_words_clean"]].describe().round(1))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df.n_chars_raw.clip(upper=df.n_chars_raw.quantile(.99)), bins=60, color="#3949ab", ax=axes[0])
axes[0].set_title("Raw Document Length (characters, clipped at 99th pct)"); axes[0].set_xlabel("Characters"); axes[0].set_ylabel("Documents")
sns.histplot(df.n_words_clean.clip(upper=df.n_words_clean.quantile(.99)), bins=60, color="#e65100", ax=axes[1])
axes[1].set_title("Word Count of Cleaned Text (clipped at 99th pct)"); axes[1].set_xlabel("Words"); axes[1].set_ylabel("Documents")
plt.tight_layout(); fig.savefig(FIG / "eda_length_distributions.png", dpi=150); plt.show()

## Category-wise document length

In [ ]:
order = df.groupby("category").n_words_clean.median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 8))
sns.boxplot(data=df, x="n_words_clean", y="category", order=order, showfliers=False, color="#9fa8da", ax=ax)
ax.set_title("Word Count by Category (outliers hidden)"); ax.set_xlabel("Words in cleaned text"); ax.set_ylabel("Category")
plt.tight_layout(); fig.savefig(FIG / "eda_length_by_category.png", dpi=150); plt.show()

## Most frequent words

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
train = df[df.dataset_split == "train"]
cnt = Counter(w for t in train.cleaned_text for w in t.split() if len(w) > 1 and w not in ENGLISH_STOP_WORDS)
top = pd.DataFrame(cnt.most_common(30), columns=["word","count"])
fig, ax = plt.subplots(figsize=(9, 8))
sns.barplot(data=top, x="count", y="word", color="#3949ab", ax=ax)
ax.set_title("30 Most Frequent Words (training split, stopwords removed)"); ax.set_xlabel("Frequency"); ax.set_ylabel("Word")
plt.tight_layout(); fig.savefig(FIG / "eda_top_words.png", dpi=150); plt.show()

In [ ]:
# Most frequent words in four example categories
cats = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.guns"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, c in zip(axes.ravel(), cats):
    cc = Counter(w for t in train[train.category == c].cleaned_text for w in t.split() if len(w) > 1 and w not in ENGLISH_STOP_WORDS)
    d = pd.DataFrame(cc.most_common(12), columns=["word","count"])
    sns.barplot(data=d, x="count", y="word", color="#3949ab", ax=ax)
    ax.set_title(c); ax.set_xlabel("Frequency"); ax.set_ylabel("Word")
fig.suptitle("Top Words in Selected Categories", y=1.01)
plt.tight_layout(); fig.savefig(FIG / "eda_top_words_by_category.png", dpi=150, bbox_inches="tight"); plt.show()

## Sample documents

In [ ]:
for c in ["sci.med", "rec.autos", "misc.forsale"]:
    row = train[(train.category == c) & (train.n_words_clean > 40)].iloc[0]
    print("="*90, "\nCATEGORY:", c, "| id:", row.document_id)
    print("--- RAW (first 400 chars) ---\n", row.document_text[:400])
    print("--- CLEANED (first 300 chars) ---\n", row.cleaned_text[:300])

## Observations and conclusions
Write your own observations here after running the cells above: class balance, duplicates, empty documents, length differences between categories, vocabulary overlap between related categories (e.g. the `comp.*`, `talk.politics.*` and religion groups), and the effect of removing headers/footers/quotes. Reference the actual numbers printed above.